# DREAM AI — verified self-improvement on Kaggle

This notebook runs one controlled instruction-tuning cycle on DREAM's own native checkpoint. It uses only explicitly approved SFT records, audits the candidate, evaluates it on the original held-out test corpus, compares the task diagnostic, and promotes only when the gates pass.

## Required private Kaggle Dataset input
Attach one private dataset containing:
- A previous DREAM training artifact ZIP (or extracted files) with `dream-bpe.pt` or `dream-bpe-instruct.pt`, `data/prepared/manifest.json`, and `data/prepared/test.txt`.
- `sft-train.jsonl` and `sft-validation.jsonl` containing separate, human-reviewed, explicitly approved instruction examples.

The SFT trainer rejects records without verified provenance, rights/privacy/quality review, and explicit approval. Do not set those flags unless the reviews were actually completed. GPU access is required for practical training and is subject to Kaggle availability.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile, shutil, json
import torch

REPO = Path('/kaggle/working/dream-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'feat/verified-self-improvement', '--single-branch', 'https://github.com/mjnr0477/dream-ai.git', str(REPO)], check=True)
os.chdir(REPO)

if not torch.cuda.is_available():
    raise RuntimeError('DREAM self-improvement requires a Kaggle GPU accelerator. Open Notebook Settings and select a GPU.')
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

INPUT_ROOT = Path('/kaggle/input')
UNPACKED = Path('/kaggle/working/dream-inputs')
UNPACKED.mkdir(parents=True, exist_ok=True)
zip_files = list(INPUT_ROOT.rglob('*.zip'))
for archive in zip_files:
    target = UNPACKED / archive.stem
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(target)

SEARCH_ROOTS = [INPUT_ROOT, UNPACKED]
def find_named(filename):
    matches = []
    for root in SEARCH_ROOTS:
        matches.extend(p for p in root.rglob(filename) if p.is_file())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise RuntimeError(f'Expected exactly one {filename}; found {len(unique)}. Check the attached private dataset.')
    return unique[0]

def find_checkpoint():
    for filename in ('dream-active.pt', 'dream-bpe-instruct.pt', 'dream-bpe.pt'):
        matches = sorted({p for root in SEARCH_ROOTS for p in root.rglob(filename) if p.is_file()})
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f'Multiple {filename} checkpoints found; attach one unambiguous active checkpoint.')
    raise RuntimeError('No DREAM checkpoint found. Attach the ZIP from the completed DREAM training notebook.')

BASE_CHECKPOINT = find_checkpoint()
MANIFEST = find_named('manifest.json')
TEST_DATA = find_named('test.txt')
SFT_TRAIN = find_named('sft-train.jsonl')
SFT_VALIDATION = find_named('sft-validation.jsonl')
print('Checkpoint:', BASE_CHECKPOINT.name)
print('Training and validation inputs found; the trainer will verify their approvals and provenance.')

## Run the verified learning cycle

The active checkpoint is written to `/kaggle/working/dream-active.pt`. If training, auditing, evaluation, or comparison fails, the previous checkpoint remains untouched.

In [ ]:
ACTIVE = Path('/kaggle/working/dream-active.pt')
command = [
    sys.executable, 'model/self_improve.py',
    '--checkpoint', str(BASE_CHECKPOINT),
    '--train-data', str(SFT_TRAIN),
    '--validation-data', str(SFT_VALIDATION),
    '--pretraining-manifest', str(MANIFEST),
    '--pretraining-test-data', str(TEST_DATA),
    '--out', str(ACTIVE),
    '--steps', '100',
    '--batch-size', '2',
    '--grad-accum-steps', '4',
    '--learning-rate', '0.00002',
]
print('Starting DREAM verified self-improvement...')
subprocess.run(command, check=True)
print('Promoted checkpoint:', ACTIVE)
print('Decision report:', str(ACTIVE) + '.self-improve.json')

## Package the result

Save a notebook version after the run. Download `dream-self-improvement-artifacts.zip` from the notebook Output panel. Keep it private because it contains model weights.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from hashlib import sha256

bundle = Path('/kaggle/working/dream-self-improvement-artifacts.zip')
entries = [
    (ACTIVE, 'checkpoints/dream-active.pt'),
    (ACTIVE.with_name(ACTIVE.name + '.self-improve.json'), 'reports/self-improvement.json'),
    (ACTIVE.with_name(ACTIVE.stem + '.previous' + ACTIVE.suffix), 'checkpoints/dream-previous.pt'),
]
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    sums = []
    for path, name in entries:
        if not path.is_file():
            raise FileNotFoundError(f'Missing expected self-improvement artifact: {path}')
        archive.write(path, arcname=name)
        sums.append(f'{sha256(path.read_bytes()).hexdigest()}  {name}')
    archive.writestr('SHA256SUMS.txt', '\\n'.join(sums) + '\\n')
print('Artifact bundle:', bundle)
print('Size (bytes):', bundle.stat().st_size)